# 01 · Dataset and parameters
**Brief: "Dataset" and Task 1 ("Load the data and set your parameters").**

Part 1 looks at the five raw CSVs. Part 2 profiles the data the way the brief asks, then sets H, W and the event.

**Data:** [Kaggle: Microsoft Azure Predictive Maintenance](https://www.kaggle.com/datasets/arnabbiswas1/microsoft-azure-predictive-maintenance),
downloaded with `kagglehub` (no account needed). **Code:** helper functions live in `../src`; a **Relies on** note
names the file and section before each step that uses one.

In [1]:
# --- Setup ----------------------------------------------------------------------------
# Libraries, plus the project's helper files in ../src (data.py, detectors.py, evaluation.py).
# Every analysis function this notebook calls lives there, in numbered sections (§1 ... §9).
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")  # harmless: kagglehub's progress bar

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_colwidth", 120, "display.width", 200)  # show long rule texts in full

ROOT = Path.cwd().parent            # project root (this notebook lives in notebooks/)
sys.path.insert(0, str(ROOT / "src"))  # so `from data import ...` finds ../src/data.py

In [2]:
# --- Scripts used in this notebook (from ../src) 
from data import (                            # src/data.py
    Config,                                   #   §1 settings: H, W, event, guardrails, seed
    load_panel,                               #   §2 read + check the CSVs, reshape into arrays
    dev_holdout_split,                        #   §2 which machines are dev / holdout
    SIGNALS, COMPONENTS,                      #   §2 the 4 sensor names and the 4 part names
)
from evaluation import (                      # src/evaluation.py
    lag_profiles, lookalike_excursions,       #   §7 what happens before failures; look-alike excursions
    learn_mapping, stamp_alignment,           #   §7 part<->signal/error pairing; hour-by-hour check around the stamp
    save_results,                             #   §9 save this notebook's key results for notebook 06
)

---
# Part 1 · Meet the raw data
The five CSVs exactly as Kaggle ships them, read with plain pandas.

| File | One row is… |
|---|---|
| `PdM_telemetry.csv` | one hourly sensor reading (voltage, rotation, pressure, vibration) for one machine |
| `PdM_errors.csv` | one error code logged by a machine's controller (the machine keeps running) |
| `PdM_maint.csv` | one component replaced, either scheduled or as the repair after a failure |
| `PdM_failures.csv` | one component that failed and had to be replaced |
| `PdM_machines.csv` | one machine: its model and age in years |

In [3]:
import kagglehub

# Download once (about 31 MB), then reuse the local cache. No Kaggle account needed.
DATA = Path(kagglehub.dataset_download("arnabbiswas1/microsoft-azure-predictive-maintenance"))
print("files in", DATA)

# Read all five files as-is. parse_dates turns the "datetime" text column into real timestamps.
raw = {name: pd.read_csv(DATA / f"PdM_{name}.csv", parse_dates=["datetime"] if name != "machines" else None)
       for name in ("telemetry", "errors", "maint", "failures", "machines")}

# One line per file: how big it is and which columns it has.
pd.DataFrame([dict(file=f"PdM_{k}.csv", rows=len(v), columns=", ".join(v.columns),
                   size_MB=round((DATA / f"PdM_{k}.csv").stat().st_size / 1e6, 1)) for k, v in raw.items()])

files in /Users/avijay/.cache/kagglehub/datasets/arnabbiswas1/microsoft-azure-predictive-maintenance/versions/3


,file,rows,columns,size_MB
0,PdM_telemetry.csv,876100,"datetime, machineID, volt, rotate, pressure, vibration",80.1
1,PdM_errors.csv,3919,"datetime, machineID, errorID",0.1
2,PdM_maint.csv,3286,"datetime, machineID, comp",0.1
3,PdM_failures.csv,761,"datetime, machineID, failure",0.0
4,PdM_machines.csv,100,"machineID, model, age",0.0


## A · Telemetry: hourly sensor readings

In [4]:
tel = raw["telemetry"]
display(tel.head())                       # what a row looks like
display(tel.describe().round(2))          # ranges: no negative or zero rotations, no absurd values

# Coverage checks: same number of rows for every machine, and readings exactly one hour apart.
per_machine = tel.groupby("machineID").size()
steps = tel.sort_values(["machineID", "datetime"]).groupby("machineID").datetime.diff().dropna()
print("rows per machine:", per_machine.min(), "to", per_machine.max())
print("time between readings:", steps.value_counts().to_dict())
print("missing values:", int(tel.isna().sum().sum()))
print("date range:", tel.datetime.min(), "->", tel.datetime.max())

,datetime,machineID,volt,rotate,pressure,vibration
0,2015-01-01 06:00:00,1,176.217853,418.504078,113.077935,45.087686
1,2015-01-01 07:00:00,1,162.879223,402.747490,95.460525,43.413973
2,2015-01-01 08:00:00,1,170.989902,527.349825,75.237905,34.178847
3,2015-01-01 09:00:00,1,162.462833,346.149335,109.248561,41.122144
4,2015-01-01 10:00:00,1,157.610021,435.376873,111.886648,25.990511


,datetime,machineID,volt,rotate,pressure,vibration
count,876100,876100.00,876100.00,876100.00,876100.00,876100.00
mean,2015-07-02 17:59:59.999999,50.50,170.78,446.61,100.86,40.39
min,2015-01-01 06:00:00,1.00,97.33,138.43,51.24,14.88
25%,2015-04-02 12:00:00,25.75,160.30,412.31,93.50,36.78
50%,2015-07-02 18:00:00,50.50,170.61,447.56,100.43,40.24
75%,2015-10-02 00:00:00,75.25,181.00,482.18,107.56,43.78
max,2016-01-01 06:00:00,100.00,255.12,695.02,185.95,76.79
std,NaN,28.87,15.51,52.67,11.05,5.37


rows per machine: 8761 to 8761
time between readings: {Timedelta('0 days 01:00:00'): 876000}
missing values: 0
date range: 2015-01-01 06:00:00 -> 2016-01-01 06:00:00


**Conclusion.** 100 machines × 8,761 readings, exactly 1 h apart, no missing values, all of 2015.

## B · Errors: controller fault codes
An error is logged while the machine keeps running: a symptom, not a breakdown.

In [5]:
err = raw["errors"]
display(err.head())                                            # one row = one error code logged at one hour

# How often each of the five codes appears overall (error1 is the most common, error5 the rarest).
print("errors per code:", err.errorID.value_counts().sort_index().to_dict())

# How evenly errors are spread across machines: a narrow range means no machine is an extreme outlier.
per_m = err.groupby("machineID").size()
print(f"errors per machine: median {per_m.median():.0f}, range {per_m.min()}-{per_m.max()}")

# Errors per month: a narrow range means the rate is stable through the year (no drift or regime change).
monthly = err.groupby(err.datetime.dt.month).size()
print(f"errors per month: {monthly.min()}-{monthly.max()}")

,datetime,machineID,errorID
0,2015-01-03 07:00:00,1,error1
1,2015-01-03 20:00:00,1,error3
2,2015-01-04 06:00:00,1,error5
3,2015-01-10 15:00:00,1,error4
4,2015-01-22 10:00:00,1,error4


errors per code: {'error1': 1010, 'error2': 988, 'error3': 838, 'error4': 727, 'error5': 356}
errors per machine: median 39, range 22-60
errors per month: 291-368


**Conclusion.** ~3,900 codes of five types (error1 most common, error5 rarest), steady through the year (291–368 a month) and across machines (22–60 each).

## C · Maintenance: component replacements
Both **scheduled** replacements and the **repair** logged after every failure.

In [6]:
mnt = raw["maint"]
display(mnt.head())                                            # IMPT [one row = one component replaced on one machine]

# Replacements per component: roughly balanced across the four parts.
print("replacements per component:", mnt.comp.value_counts().sort_index().to_dict())

# The log starts in mid-2014, before the telemetry does. Those early rows tell me each part's age on day one.
print("date range:", mnt.datetime.min(), "->", mnt.datetime.max(),
      f"({(mnt.datetime < tel.datetime.min()).sum()} rows before telemetry starts)")

# Hour of day of every replacement: all at 06:00, i.e. the log is written once a day in a batch.
print("hour of day:", mnt.datetime.dt.hour.value_counts().to_dict(), " <- all at 06:00: a daily batch log")

# Days between consecutive replacements of the same part on the same machine ("service interval").
# This is what later becomes the "hours since last maintenance" counter.
gap = mnt.sort_values("datetime").groupby(["machineID", "comp"]).datetime.diff().dt.days.dropna()
print(f"service interval: median {gap.median():.0f} days (middle half {gap.quantile(.25):.0f}-{gap.quantile(.75):.0f})")

,datetime,machineID,comp
0,2014-06-01 06:00:00,1,comp2
1,2014-07-16 06:00:00,1,comp4
2,2014-07-31 06:00:00,1,comp3
3,2014-12-13 06:00:00,1,comp1
4,2015-01-05 06:00:00,1,comp4


replacements per component: {'comp1': 804, 'comp2': 863, 'comp3': 808, 'comp4': 811}
date range: 2014-06-01 06:00:00 -> 2016-01-01 06:00:00 (400 rows before telemetry starts)
hour of day: {6: 3286}  <- all at 06:00: a daily batch log
service interval: median 45 days (middle half 15-75)


**Conclusion.** 3,286 replacements, balanced across the four parts, all stamped 06:00 (a daily batch). The same part is replaced every 45 days (median). The log starts 6 months before the telemetry, so every part's age is known from day one.

## D · Failures: the rare events

In [7]:
fail = raw["failures"]
display(fail.head())
print("failures per component:", fail.failure.value_counts().sort_index().to_dict())
fpm = fail.groupby("machineID").size().reindex(range(1, 101), fill_value=0)
print(f"failures per machine: median {fpm.median():.0f}, range {fpm.min()}-{fpm.max()}; "
      f"{(fpm == 0).sum()} machines never fail")
print("hour of day:", fail.datetime.dt.hour.value_counts().to_dict())

# Is every failure also in the maintenance log (the repair)? Expect almost all [Yes, 743/761!]
both = fail.merge(mnt.rename(columns={"comp": "failure"}), on=["machineID", "datetime", "failure"])
print(f"failures with a matching repair in the maintenance log: {len(both)} of {len(fail)}")
unmatched = fail.merge(both[["machineID", "datetime", "failure"]], how="left", indicator=True).query("_merge == 'left_only'")
print("the ones without a repair are all stamped:", unmatched.datetime.dt.strftime("%Y-%m-%d %H:%M").value_counts().to_dict())

,datetime,machineID,failure
0,2015-01-05 06:00:00,1,comp4
1,2015-03-06 06:00:00,1,comp1
2,2015-04-20 06:00:00,1,comp2
3,2015-06-19 06:00:00,1,comp4
4,2015-09-02 06:00:00,1,comp4


failures per component: {'comp1': 192, 'comp2': 259, 'comp3': 131, 'comp4': 179}
failures per machine: median 7, range 0-19; 2 machines never fail
hour of day: {6: 743, 3: 18}
failures with a matching repair in the maintenance log: 743 of 761
the ones without a repair are all stamped: {'2015-01-02 03:00': 18}


**Conclusion.** 761 failures: comp2 most (259), comp3 least (131); 0–19 per machine, 2 machines never fail. 743 are stamped 06:00 with a matching repair; the other 18 are one 03:00 batch on 2 Jan 2015.

## E · Machines: model and age

In [8]:
mach = raw["machines"]
display(mach.head())                                           # one row = one machine: its model and age

# Four machine models; model3 and model4 make up two thirds of the fleet. Age and model are the only
# "operating regime" information; Part 2 checks whether signal levels differ by model.
print("machines per model:", mach.model.value_counts().sort_index().to_dict())
print(f"age: {mach.age.min()}-{mach.age.max()} years (median {mach.age.median():.0f})")

,machineID,model,age
0,1,model3,18
1,2,model4,7
2,3,model3,8
3,4,model3,7
4,5,model3,2


machines per model: {'model1': 16, 'model2': 17, 'model3': 35, 'model4': 32}
age: 0-20 years (median 12)


**Conclusion.** Four models (16, 17, 35 and 32 machines), aged 0–20 years (median 12).

## F · How the files connect
Every file shares **`machineID`**. The event files also share **`datetime`**, on the same hourly grid as the
telemetry. Failures join to maintenance on machine + hour + component (the repair). One machine's year shows it all.

In [9]:
# One machine's year: a sensor trace with its errors, replacements and failures marked on top.
# Pick the machine with the most failures so there is plenty to see [makes life easier for us!]
mid = int(fail.machineID.value_counts().idxmax())
t = tel[tel.machineID == mid].set_index("datetime")
day = t.vibration.resample("D").mean()                 # daily mean, so the year fits on one chart

fig = go.Figure()
fig.add_scatter(x=day.index, y=day.values, name="vibration (daily mean)", line=dict(color="gray"))
top = day.max() * 1.05
for df, label, sym, col in ((err[err.machineID == mid], "error", "line-ns", "#b76e00"),
                            (mnt[mnt.machineID == mid], "replacement", "triangle-up", "#0e6170"),
                            (fail[fail.machineID == mid], "failure", "x", "#b03a3a")):
    fig.add_scatter(x=df.datetime, y=[top] * len(df), mode="markers", name=label,
                    marker=dict(symbol=sym, size=9, color=col, line=dict(width=1.5, color=col)),
                    hovertext=df.iloc[:, 2].astype(str))   # error code / component on hover
fig.update_layout(title=f"Machine {mid}: one year of vibration with its errors, replacements and failures",
                  yaxis_title="vibration", height=420)
fig   # look for: errors cluster just before failures; every failure has a replacement at the same moment

**Conclusion.** Errors bunch up in the days before failures, and every failure has a replacement at the same
moment. So the replacement is part of the failure, not a warning sign: using it would be leakage. The data is
unusually regular (06:00 stamps, no gaps, even rates), so it is probably synthetic: expect cleaner results than
on a real line.

---
# Part 2 · Profile for the analysis
Part 1 looked at the CSVs as shipped. From here on, the data is loaded into analysis-ready arrays and
profiled against what the brief asks: events, sampling, gaps, regimes, counters, and the choice of H and W.

### H and W in plain terms
* **H (lead time): how much warning you get.** The gap between the alarm and the failure. With H = 12, a
  warning sign only counts if it comes at least 12 hours before the failure, leaving half a day to act.
* **W (window length): how far back you look.** After skipping back H hours, W is how long a stretch is
  searched for suspicious behaviour. With W = 36, that is the 36 hours before that.

For a failure at time T, I look for warning signs between **T − (H + W)** and **T − H**. With the defaults,
that is 48 to 12 hours before the failure:

```
  ◄─────────── W = 36h ───────────►◄── H = 12h ──►
  |          look for signs         |  time to    |  failure
 T-48h                            T-12h   act      T
```

Think of a weather forecast: H is how far ahead the warning must arrive to be useful (time to bring the
washing in); W is how many hours of sky-watching the forecast is based on. Section 3 checks how failures are
timestamped, and section 6 shows where the warning signs live (each part's signal starts drifting ~48 h
before failure). 

> **Relies on:**  
> · `src/data.py` §1 — settings (`Config`)  
> · `src/data.py` §2 — loading the data (`load_panel`)

## 1 · Load the data

In [10]:
panel = load_panel()   # downloads on first run (cached afterwards), then checks the grid with asserts
cfg = Config()         # default settings: H=12, W=36, recovery 24h, seed 7

# Size and time span. Expect 100 machines x 8,761 hourly readings = one calendar year.
print(f"{panel.M} machines x {panel.T:,} hours = {panel.M*panel.T:,} telemetry rows")
print("span:", panel.times[0], "->", panel.times[-1])
print("sampling: every 1 hour; gaps: none (load_panel asserted it)")

100 machines x 8,761 hours = 876,100 telemetry rows
span: 2015-01-01 06:00:00 -> 2016-01-01 06:00:00
sampling: every 1 hour; gaps: none (load_panel asserted it)


## 2 · How many events?
The brief says failures on the **same machine at the same timestamp** count as **one** event
(e.g. comp1 and comp3 replaced together). `load_panel` already merged them; here I check the counts.

In [11]:
fr, ev = panel.failures_raw, panel.events   # raw failure rows vs. merged events

# 761 raw failure rows collapse into fewer events, because some failures share a machine and an hour.
print("failure rows:", len(fr), "-> events:", len(ev),
      "| events where 2+ parts failed together:", (ev.n_comps > 1).sum())

# Events per component. A multi-part event counts for each part it involves, so these sum to more than len(ev).
per_part = pd.Series({c: ev.comps.map(lambda x: c in x).sum() for c in COMPONENTS}, name="events involving")
per_part

failure rows: 761 -> events: 719 | events where 2+ parts failed together: 42


comp1    192
comp2    259
comp3    131
comp4    179
Name: events involving, dtype: int64

**Conclusion.** 761 failure rows become **719 events**: 42 events had two parts fail together. Per part: comp1 192, comp2 259, comp3 131, comp4 179.

## 3 · How are events timestamped? (this sets H)
Failures and repairs are stamped at 06:00, a daily batch. If a failure were really logged up to a day late, the
hours just before its stamp would hold *post-failure* data, and scoring them would be leakage (forcing H ≥ 24 h).
The check: average each part's own signal and own error codes hour by hour, from 49 h before the stamp to 1 h after.

In [12]:
# Which signal and which error codes belong to which part, learned from the dev machines only
# (src/evaluation.py §7, learn_mapping). E.g. comp1 -> volt + error1.
dev, hold = dev_holdout_split(panel, cfg.dev_fraction, cfg.seed)
mapping = learn_mapping(panel, cfg, dev)

# Hour-by-hour averages around the stamp (src/evaluation.py §7, stamp_alignment).
# own_signal_z: the part's own signal vs its machine's normal operation (signed: a precursor is positive)
# own_error_codes_per_failure: how many of the part's own error codes are logged at that hour, per failure
S = stamp_alignment(panel, cfg, mapping)

fig = go.Figure()
for c, d in S.items():
    fig.add_scatter(x=d.index, y=d.own_signal_z, name=f"{c}: own signal (z)")
fig.add_vline(x=0, line_dash="dot", annotation_text="failure stamp")
fig.update_xaxes(autorange="reversed", title="hours before the stamp (negative = after)")
fig.update_layout(yaxis_title="z vs own normal", height=360,
                  title="Each part's own signal, hour by hour around its failure stamp")
fig.show()

# The evidence in one table, per part: is the signal still shifted at the stamp, and back to normal 1 h after?
# How many of the part's own error codes appear in the final 23 h, and how many exactly 24 h before?
stamp = pd.DataFrame({c: {"signal z at the stamp": d.own_signal_z[0], "signal z 1 h after": d.own_signal_z[-1],
                  "own codes in the last 23 h": d.own_error_codes_per_failure.loc[1:23].sum(),
                  "own codes exactly 24 h before": d.own_error_codes_per_failure[24]}
              for c, d in S.items()}).T.round(2)
stamp

,signal z at the stamp,signal z 1 h after,own codes in the last 23 h,own codes exactly 24 h before
comp1,1.27,0.10,0.02,1.0
comp2,1.48,-0.13,0.02,2.0
comp3,2.31,-0.06,0.02,1.0
comp4,1.88,-0.06,0.01,1.0


**Conclusion.** The stamp is a daily batch, but **no post-failure data sits before it**. Each part's signal stays
shifted right up to the stamp and is back to normal 1 h after it. Each part's own error code appears exactly 24 h
before the stamp, with nothing in the final 23 h. So H isn't forced to 24 h: it is set to **H = 12** so that
every alert that counts leaves half a day to act, and notebook 04 checks it on an F1 grid.

## 4 · Gaps, clustering and operating regimes
Three checks: how far apart failures on one machine are, whether machines ever stop, and whether machine models
run at different levels.

In [13]:
# Days between consecutive events on the same machine.
gaps = ev.sort_values(["m", "t"]).groupby("m").t.diff().dropna() / 24
print("days between events on a machine:", gaps.describe().round(1).to_dict())
print("events within 7 days of the previous one:", (gaps < 7).sum(), " <- these windows can overlap")

# If rotation ever hit 0, the machine would be stopped and those hours could not be "normal".
print("min rotation:", panel.X["rotate"].min().round(1), "rpm  -> never stops, so every hour counts as running")

# Mean signal level per machine model. Big differences would argue for per-model thresholds.
reg = panel.machines.groupby("model").agg(machines=("machineID", "size"), mean_age=("age", "mean"))
models = panel.machines.model.to_numpy()
for s in SIGNALS:
    reg[s] = [panel.X[s][models == m].mean().round(2) for m in reg.index]
reg   # expect nearly identical levels: regimes barely differ in this (synthetic) data

days between events on a machine: {'count': 621.0, 'mean': 46.1, 'std': 38.3, 'min': 0.1, '25%': 15.0, '50%': 30.0, '75%': 60.0, 'max': 270.0}
events within 7 days of the previous one: 3  <- these windows can overlap
min rotation: 138.4 rpm  -> never stops, so every hour counts as running


,machines,mean_age,volt,rotate,pressure,vibration
model,,,,,,
model1,16,12.250000,170.78,446.37,101.27,40.44
model2,17,12.764706,170.72,446.81,101.22,40.39
model3,35,12.028571,170.81,446.51,100.67,40.40
model4,32,9.343750,170.77,446.71,100.67,40.34


**Conclusion.**
* **Gaps:** none. Every machine has a reading every hour of 2015, with no missing values. Failures on the same
  machine are a median **30 days** apart (shortest 0.1 days, longest 270).
* **Clustering:** **not a factor here.** Only **3** of 719 events come within a week of the previous one on the same
  machine, so clustering in time changes nothing in the analysis. Failures do vary *by machine* (0 to 19 per
  machine), so confidence intervals resample whole machines.
* **Operating regimes:** effectively **one**. The four models run at the same levels (voltage 170.7–170.8,
  rotation 446–447, pressure 100.7–101.3, vibration 40.3–40.4 for every model). Machines never stop (lowest
  rotation 138 rpm), so there is no running/idle split. Per-model and per-machine thresholds are still tried in
  notebook 04, in case.

## 5 · Counters vs. live readings
A **live reading** wanders up and down from hour to hour. A **counter** climbs by exactly +1 an hour, then resets.
The test: how often each series goes up, and how often by exactly +1.

In [14]:
def describe(x, name):
    """Share of hourly steps that go up, that are exactly +1, and that don't change."""
    d = np.diff(x, axis=1)        # hour-to-hour change, per machine
    d = d[np.isfinite(d)]          # ignore NaN (before the first maintenance record)
    return dict(series=name,
                up=(d > 0).mean().round(3),          # ~0.5 for a live reading
                plus_one=(d == 1).mean().round(3),   # ~1.0 for an hourly counter
                unchanged=(d == 0).mean().round(3))

# Expect: the four sensors are live readings (up ~50%, never exactly +1);
# hours-since-maintenance is a counter (+1 almost every hour).
counters = pd.DataFrame([describe(panel.X[s], s) for s in SIGNALS]
                        + [describe(panel.hours_since_maint[:, :, 0], "hours since maint (comp1)")])
counters

,series,up,plus_one,unchanged
0,volt,0.500,0.000,0.0
1,rotate,0.500,0.000,0.0
2,pressure,0.500,0.000,0.0
3,vibration,0.500,0.000,0.0
4,hours since maint (comp1),0.999,0.999,0.0


**Conclusion.**

| Series | Type | Evidence |
|---|---|---|
| `volt`, `rotate`, `pressure`, `vibration` (the telemetry) | **live reading** | up 50% of hours, down the other 50%, never exactly +1 |
| hours since maintenance (one per part, built from `PdM_maint.csv`) | **counter** | +1 in 99.9% of hours, reset to 0 at each replacement |

The raw files contain no counter; the only one is the "hours since maintenance" I build. Counters need care,
because their "normal" range is set by the maintenance schedule, not by machine health (notebook 05, section 3).

## 6 · What happens before a failure? (this sets W)
Line up every failure of one part at lag 0 and average each signal 1, 2, 3, … hours before it. Signals are
z-scores against each machine's own normal operation (1.5 = 1.5 standard deviations above its usual level).

> **Relies on:**  
> · `src/data.py` §3 — event windows and normal operation (`build_masks`)  
> · `src/evaluation.py` §7 — event-triggered averages (`lag_profiles`)

In [15]:
L = lag_profiles(panel, cfg)   # mean z-score of each signal at each lag (0..168 h) before each part's failures

fig = go.Figure()
for c in COMPONENTS:
    # For each component, plot only its "signature" signal: the one that moves most before its failures.
    s = max(SIGNALS, key=lambda s: abs(np.mean(L["signals"][c][s][:48])))
    fig.add_scatter(x=L["lags"], y=L["signals"][c][s], name=f"{c}: {s}")

# Shade the scored window: lags max(H,1) .. H+W-1 before the failure (12..47 h with the defaults).
fig.add_vrect(x0=max(cfg.H, 1), x1=cfg.H + cfg.W - 1, opacity=.15, line_width=0, annotation_text="scored window")
fig.update_xaxes(autorange="reversed", title="hours before failure").update_yaxes(title="z vs own normal")
fig   # look for: flat at 0, then a step up (or down) about 48 h before the failure

**Conclusion.** Each part has one signature signal: comp1 → volt, comp2 → rotate, comp3 → pressure,
comp4 → vibration. It is flat at normal for the whole week before, then shifts **in a single step exactly 48 h
before the failure** and stays shifted until it. So the window should reach back exactly 48 h: **H + W = 48**.
Shorter would cut off the start of the drift; longer adds nothing but lets unrelated earlier alerts count as real.
With H = 12 (section 3) that leaves **W = 36** (lags 12–47). Notebook 04, section 6 checks the split on an F1 grid.
*A perfectly sharp step is a sign of simulated data: on a real line this lead would be measured and would vary.*

## 7 · The key property: look-alike excursions
If every signal shift ended in a failure, a signal threshold would be a perfect detector. So count every
sustained excursion of the same size and length, and how many were followed by a failure.

**What a "drift" means here, in plain terms.** Every machine has its own normal range for each sensor, the way each
person has their own resting heart rate. A *drift* is when a part's sensor moves noticeably outside that machine's
normal range and **stays there for at least a full day**; a single odd reading doesn't count. Precisely: the part's
own sensor as a z-score against that machine's normal hours, averaged over 12 h, staying beyond half the part's
typical pre-failure shift (0.7 to 1.2 standard deviations) for at least 24 h. It is *real* if that part failed
between 12 h before and 24 h after the drift ended; otherwise it is a *look-alike*.

> **Relies on:** `src/evaluation.py` §7 — `lookalike_excursions` (cuts out every sustained excursion of each
> part's signature signal and checks whether a failure followed and whether an error code was logged during it).

In [16]:
LA = lookalike_excursions(panel, cfg)   # one row per component
look = pd.DataFrame(LA).T[["signal",                          # signature signal, found from the data
                    "episodes_ending_in_failure",      # excursions followed by a failure of this part
                    "episodes_without_failure",        # same-looking excursions with no failure after
                    "precision_ceiling",               # best possible precision of a signal-only rule
                    "share_with_error_failure",        # real excursions that contain an error code
                    "share_with_error_none",           # look-alikes that contain an error code
                    "precision_ceiling_with_error"]]   # ceiling if I also require an error
look

,signal,episodes_ending_in_failure,episodes_without_failure,precision_ceiling,share_with_error_failure,share_with_error_none,precision_ceiling_with_error
comp1,volt,172,464,0.27,0.959,0.144,0.711
comp2,rotate,248,475,0.343,0.992,0.183,0.739
comp3,pressure,126,483,0.207,1.0,0.153,0.63
comp4,vibration,171,476,0.264,1.0,0.153,0.701


**Conclusion.** Each part's signal drifts like this **~470 times without a failure**, against 126–248 times before
one. So a signal-only rule is right at most **21–34%** of the time. The difference is the error code: an error code
(any code) is logged during **96–100%** of real pre-failure drifts but only **14–18%** of look-alikes. Pooled over
the four parts, requiring one lifts the share of drifts that are real from **27% to 70%** while keeping 99% of real
drifts. This is why the best detectors pair a signal with an error code.

> **In one line:** A drift is a sensor running a fever: off its own normal and staying there for a day. Most fevers
> pass on their own. The error code tells us which ones are serious.

## 8 · Task 1: setting the parameters
Every setting lives in one object, `Config` (`src/data.py` §1). H, W and the event are changed in one line,
e.g. `Config(H=24, W=24, event="comp2")`. Use the same settings in every notebook.

In [17]:
# The settings used throughout. Defaults: H=12, W=36, event="any" (any component failure).
cfg = Config()
print(f"H = {cfg.H} h   (lead time: the scoring window ends H hours before each event)")
print(f"W = {cfg.W} h   (window length: the window covers lags H .. H+W-1 hours before the event)")
print(f"event = {cfg.event!r}   (options: 'any', 'comp1', 'comp2', 'comp3', 'comp4')")
print(f"recovery guardrail = {cfg.recovery_h} h after any failure or repair is not 'normal'; warm-up = {cfg.warmup_h} h")

# Example: comp2 failures with the earlier default (24 h lead, 24 h window). Nothing else needs to change.
print("\nexample of another configuration:", cfg.with_(event="comp2", H=24, W=24))

H = 12 h   (lead time: the scoring window ends H hours before each event)
W = 36 h   (window length: the window covers lags H .. H+W-1 hours before the event)
event = 'any'   (options: 'any', 'comp1', 'comp2', 'comp3', 'comp4')
recovery guardrail = 24 h after any failure or repair is not 'normal'; warm-up = 24 h

example of another configuration: Config(H=24, W=24, event='comp2', recovery_h=24, warmup_h=24, score_event_hour=False, dev_fraction=0.5, seed=7, n_bootstrap=1000, n_null_shifts=500, percentiles=(95.0, 99.0, 99.5, 99.9, 99.99))


## 9 · How confident can I be?
Every reported number comes from the **50 holdout machines** only, split five ways (any failure, four parts).
How many events does each number rest on?

In [18]:
# Same machine split as notebooks 04 and 05: 50 dev machines (used to choose) and 50 holdout (used to report).
dev, hold = dev_holdout_split(panel, cfg.dev_fraction, cfg.seed)
on_hold = panel.events.m.isin(hold)

rows = []
for kind in ["any", *COMPONENTS]:
    is_kind = panel.events.comps.map(lambda c: kind == "any" or kind in c)
    n_hold = int((is_kind & on_hold).sum())
    rows.append(dict(event=kind,
                     events_total=int(is_kind.sum()),
                     events_on_holdout=n_hold,
                     # Worst-case 95% interval half-width for a proportion (precision or coverage near 50%),
                     # treating events as independent. The real intervals (cluster bootstrap over machines,
                     # in notebook 04) are a bit wider because events cluster on machines.
                     worst_case_ci_half_width=f"±{1.96 * (0.25 / n_hold) ** 0.5:.1%}"))
conf = pd.DataFrame(rows)
conf

,event,events_total,events_on_holdout,worst_case_ci_half_width
0,any,719,363,±5.1%
1,comp1,192,90,±10.3%
2,comp2,259,135,±8.4%
3,comp3,131,65,±12.2%
4,comp4,179,91,±10.3%


**Conclusion.** "Any failure" rests on **363 holdout events**, so precision and coverage are pinned to about
±5 points. The rarest part, comp3, has 65 (±12 points): enough to tell a strong detector from a weak one, not
to split close ones. The bigger risk is that the data is synthetic and very clean (shifts at exactly 48 h, error
codes at exactly 24 h), so absolute numbers are optimistic; the method and the ranking should transfer better.

## Notebook summary
| Characteristic | Finding |
|---|---|
| Events | 761 failure rows → **719 events** (same machine + hour merged); 131–259 per part |
| Sampling and gaps | hourly, 100 machines × 8,761 h (all of 2015); no gaps, no missing values |
| Clustering | **not a factor**: 3 of 719 events within a week of the previous one; failures vary by machine (0–19 each) |
| Regimes | one: four models at the same signal levels; machines never stop |
| Counters | none in the raw data; "hours since maintenance" (built from the maintenance log) is one |
| Timestamps | daily 06:00 batch, but no post-failure data before the stamp |
| Precursor timing | each part's signal shifts in one step exactly 48 h before failure; its own error code lands exactly 24 h before |
| H and W | **H + W = 48** (the precursor's span), split as **H = 12, W = 36**: half a day of warning (checked in notebook 04, section 6) |
| Key property | ~470 look-alike drifts per part; requiring an error code lifts the share of real drifts from 27% to 70% |
| Confidence | 363 holdout events for "any failure" (±5 points), 65 for comp3 (±12); synthetic data, so optimistic |

## 10 · Save this notebook's results
The key results above are saved to `outputs/01_dataset_and_parameters.json`, so that notebook 06 (the write-up)
can summarise them without re-running anything.

In [19]:
save_results("01_dataset_and_parameters", dict(            # src/evaluation.py §9
    events=dict(failure_rows=len(fr), events=len(ev), multi_part=int((ev.n_comps > 1).sum()), per_part=per_part),
    sampling=dict(machines=panel.M, hours=panel.T, start=str(panel.times[0]), end=str(panel.times[-1])),
    gaps=dict(median_days_between_events=float(gaps.median()), events_within_7_days=int((gaps < 7).sum()),
              min_rotation_rpm=float(panel.X["rotate"].min())),
    regimes=reg,              # mean signal level per machine model (section 4)
    counters=counters,        # live readings vs counters (section 5)
    stamp=stamp,              # the timestamp audit (section 3)
    lookalikes=look,          # look-alike excursions (section 7)
    confidence=conf,          # holdout events per event type (section 9)
    mapping=mapping,          # each part's own signal and error codes, learned on dev machines
), cfg)

saved outputs/01_dataset_and_parameters.json


PosixPath('/Users/avijay/Documents/oden_precursors/outputs/01_dataset_and_parameters.json')